In [ ]:
import torch
from torch import nn
import matplotlib.pyplot as plt
from src.model import NeuralNetwork
from src.dataset import train_dataloader, validation_dataloader, training_data

In [ ]:
# Configurable parameters
learning_rate = 1e-4
batch_size = 24
target_epoch = 130   # set epoch where you want to reach
patience = 10
save = True
best_model_weights_path = "fc_model_weights.pth"
best_model_full_path = "fc_model.pth"

In [ ]:
# Initialize model, loss function and optimizer
model = NeuralNetwork()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

best_val_accuracy = 0.0
epochs_without_improvement = 0
start_epoch = 0

if save:
    try:
        checkpoint = torch.load(best_model_weights_path)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        best_val_accuracy = checkpoint['best_val_accuracy']
        start_epoch = checkpoint['epoch']
        epochs_without_improvement = checkpoint['epochs_without_improvement']
        print(f"Resuming from epoch {start_epoch} with best validation accuracy {(100*best_val_accuracy):>0.1f}%")
    except:
        print("No checkpoint found, starting fresh")

In [ ]:
# Training loop
def train_loop(dataloader, model, loss_fn, optimizer):
    model.train()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    train_loss = 0
    correct = 0
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_loss += loss.item()
        correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    train_loss /= num_batches
    correct /= size
    print(f"Training accuracy: {(100*correct):>0.1f}%, Training avg loss: {train_loss:>8f}")
    return train_loss

# Validation loop
def test_loop(dataloader, model, loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Validation accuracy: {(100*correct):>0.1f}%, Validation avg loss: {test_loss:>8f}")
    return test_loss, correct

In [ ]:
# Train the model
train_losses = []
test_losses = []
best_weights = None

for epoch in range(start_epoch + 1, target_epoch + 1):
    print(f"Epoch {epoch}")
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loss, val_accuracy = test_loop(validation_dataloader, model, loss_fn)
    train_losses.append(train_loss)
    test_losses.append(test_loss)

    # Save best weights in memory if validation accuracy improved
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_without_improvement = 0
        best_epoch = epoch
        best_weights = {
            'epoch': epoch,
            'model_state': model.state_dict().copy(),
            'optimizer_state': optimizer.state_dict().copy(),
            'best_val_accuracy': best_val_accuracy,
            'epochs_without_improvement': 0
        }
        print(f"New best at epoch {epoch}: {(100*best_val_accuracy):>0.1f}%")
    else:
        epochs_without_improvement += 1
        print(f"No improvement for {epochs_without_improvement}/{patience} epochs")

    # Early stopping if no improvement for patience epochs
    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

# Save best model to disk only at end of training
if save and best_weights is not None:
    torch.save(best_weights, best_model_weights_path)
    model.load_state_dict(best_weights['model_state'])
    torch.save(model, best_model_full_path)
    print(f"Saved best model from epoch {best_epoch} with validation accuracy {(100*best_val_accuracy):>0.1f}%")

# Plot loss curve
plt.plot(train_losses, label="Train Loss")
plt.plot(test_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()
plt.show()

print(f"Best validation accuracy: {(100*best_val_accuracy):>0.1f}%")